# Defining models

In [ ]:
#defining classifiers with pipelining 
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Liberation Serif"], # Matches Times New Roman
    "font.size": 10,                    # Standard for most papers
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 9,
    "text.usetex": False,               
    "pdf.fonttype": 42,                 
    "ps.fonttype": 42
})

models= {
    "XGBoost": Pipeline([
        ("scaler", StandardScaler()), 
        ("clf", XGBClassifier(n_estimators=50, learning_rate=0.03, random_state=42, eval_metric='logloss'))
    ]),
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=10000, solver='lbfgs', C=1.0, random_state=42, n_jobs=-1))
    ]),
    "Naive Bayes": Pipeline([
        ("scaler", StandardScaler()), 
        ("clf", GaussianNB())
    ]),
    "Linear SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel='linear', probability=True, random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("scaler", StandardScaler()), 
        ("clf", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
    ]),
}

# BOOTSTRAPPED 95% - ROC CURVE

In [ ]:
def bootstrap_auc_ci(y_true, y_pred_proba, n_boot=1000, ci=95):
    rng = np.random.RandomState(42)
    aucs = []
    N = len(y_true)
    for _ in range(n_boot):
        indices = rng.choice(np.arange(N), N, replace=True)
        if len(np.unique(y_true[indices])) == 2:  # avoid degenerate class sampling
            aucs.append(roc_auc_score(y_true[indices], y_pred_proba[indices]))
    lower = np.percentile(aucs, (100-ci)/2)
    upper = np.percentile(aucs, 100-(100-ci)/2)
    mean_auc = np.mean(aucs)
    print(f"AUC: {mean_auc:.3f} [{lower:.3f}, {upper:.3f}] (bootstrapped {n_boot} samples)")
    return mean_auc, lower, upper

In [ ]:
# Deep Features using LR
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_curve, roc_auc_score


deep_path = "/kaggle/input/deep-features/deep_features.csv"
hand_path = "/kaggle/input/handcrafted-features/Handcrafted_Features_flair.csv"

X_deep = pd.read_csv(deep_path)
X_handcrafted = pd.read_csv(hand_path)


key_col = "id"       
label_col = "label"


y = X_handcrafted[label_col].to_numpy().ravel()


if key_col in X_deep.columns:
    X_deep_agg = X_deep.groupby(key_col).mean().reset_index()
else:
    X_deep[key_col] = np.arange(len(X_deep))
    X_deep_agg = X_deep.copy()


X_merged = pd.merge(X_deep_agg, X_handcrafted[[key_col, label_col]], on=key_col)
X_deep_final = X_merged.drop(columns=[key_col, label_col])
y_final = X_merged[label_col].to_numpy().ravel()

print(" Deep features aligned.")
print(f"Deep features shape: {X_deep_final.shape}")
print(f"Labels shape: {y_final.shape}")
print(f"Class distribution: {np.unique(y_final, return_counts=True)}")


clf =models['Logistic Regression']


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_probs = cross_val_predict(clf, X_deep_final, y_final, cv=cv, method='predict_proba')[:, 1]

fpr, tpr, _ = roc_curve(y_final, y_probs)
auc_score = roc_auc_score(y_final, y_probs)

mean_auc, lower_ci, upper_ci = bootstrap_auc_ci(y_final, y_probs, n_boot=1000, ci=95)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'Logistic Regression (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Deep Features (LR)')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()


In [ ]:
# Handcrafted Features using Random Forest

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_curve, roc_auc_score


hand_path="/kaggle/input/handcrafted-features/Handcrafted_Features_flair.csv"
X_handcrafted = pd.read_csv(hand_path)


label_col = "label"
y = X_handcrafted[label_col].to_numpy().ravel()

X_handcrafted_clean = X_handcrafted.drop(columns=[label_col], errors="ignore")


feature_cols = [c for c in X_handcrafted_clean.columns if c.lower() != "id"]
X_train = X_handcrafted_clean[feature_cols]

print("Handcrafted features loaded & cleaned.")
print(f"Training feature shape (without id): {X_train.shape}")
print(f"Original df shape (with id): {X_handcrafted_clean.shape}")
print(f"Labels shape: {y.shape}")
print(f"Class distribution: {np.unique(y, return_counts=True)}")


clf = models['Random Forest']

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_probs = cross_val_predict(clf, X_train, y, cv=cv, method='predict_proba')[:, 1]


fpr, tpr, _ = roc_curve(y, y_probs)
auc_score = roc_auc_score(y, y_probs)

mean_auc, lower_ci, upper_ci = bootstrap_auc_ci(y, y_probs, n_boot=1000, ci=95)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Random Forest (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Handcrafted Features (Random Forest)')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()


In [ ]:
# Robust Fusion ROC (Deep + Handcrafted) using LR

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_curve, roc_auc_score
from xgboost import XGBClassifier

deep_path = "/kaggle/input/deep-features/deep_features.csv"
hand_path = "/kaggle/input/handcrafted-features/Handcrafted_Features_flair.csv"

X_deep = pd.read_csv(deep_path)
X_hand = pd.read_csv(hand_path)

print("Loaded shapes -> deep:", X_deep.shape, ", handcrafted:", X_hand.shape)

key_col = "id"
label_col = "label"

print("\nHandcrafted columns (tail):", X_hand.columns[-10:].tolist())
print("Deep columns (tail):", X_deep.columns[-10:].tolist())

if label_col not in X_hand.columns:
    raise RuntimeError(f"Label column '{label_col}' not found in handcrafted file. Columns: {X_hand.columns.tolist()}")

label_vals = X_hand[label_col].unique()
print("\nLabel unique values:", label_vals)
print("Label counts:\n", X_hand[label_col].value_counts(dropna=False))


y_series = X_hand[label_col].copy()

try:
    y_numeric = pd.to_numeric(y_series, errors='coerce')
    if set(y_numeric.dropna().unique()).issubset({0,1}):
        y_encoded = y_numeric.fillna(y_numeric)  # keep numeric 0/1
        print("Labels are numeric 0/1.")
    else:
        # fallback to mapping strings
        raise ValueError
except Exception:
    # map common strings
    unique_vals = list(pd.Categorical(y_series).categories)
    unique_vals = [str(v) for v in unique_vals]
    print("Detected label categories (strings):", unique_vals)
    if len(unique_vals) == 2:
        # prefer mapping 'HGG'->1 if present
        if any(s.upper() == "HGG" for s in unique_vals):
            pos = [s for s in unique_vals if s.upper() == "HGG"][0]
            mapping = {unique_vals[0]: 1 if unique_vals[0]==pos else 0,
                       unique_vals[1]: 1 if unique_vals[1]==pos else 0}
        elif any(s.upper() == "YES" for s in unique_vals) or any(s.upper()=="POS" for s in unique_vals):
            # heuristic
            mapping = {unique_vals[0]:1, unique_vals[1]:0}
        else:
            # map second category to 1 (arbitrary but consistent)
            mapping = {unique_vals[0]:0, unique_vals[1]:1}
        print("Mapping used for labels:", mapping)
        y_encoded = y_series.map(mapping).astype(int)
    else:
        raise RuntimeError(f"Unexpected number of label categories: {len(unique_vals)} - cannot auto-encode.")

# Final check
if y_encoded.isnull().any():
    raise RuntimeError("After encoding, some labels are NaN. Inspect 'label' column values.")

if key_col in X_deep.columns:
    n_ids = X_deep[key_col].nunique()
    n_rows = X_deep.shape[0]
    print(f"\nDeep has {n_rows} rows and {n_ids} unique ids.")
    if n_rows != n_ids:
        print("Aggregating deep features by id (mean).")
        X_deep_agg = X_deep.groupby(key_col).mean().reset_index()
    else:
        X_deep_agg = X_deep.copy()
else:
    raise RuntimeError(f"Key column '{key_col}' not found in deep features. Columns: {X_deep.columns.tolist()}")

print("Deep (agg) shape:", X_deep_agg.shape)

if key_col not in X_hand.columns:
    raise RuntimeError(f"Key column '{key_col}' not found in handcrafted data. Columns: {X_hand.columns.tolist()}")

hand_for_merge = X_hand[[key_col, label_col]]


X_merged = pd.merge(hand_for_merge, X_deep_agg, on=key_col, how='inner')
print("\nAfter merge (hand w/ deep agg):", X_merged.shape)


y_final = X_merged[label_col].copy()

if pd.api.types.is_numeric_dtype(y_final):
    if set(y_final.unique()).issubset({0,1}):
        y_num = y_final.astype(int).to_numpy()
    else:
        y_num = pd.to_numeric(y_final, errors='coerce').astype(int).to_numpy()
else:
    uniq = list(pd.Categorical(y_final).categories)
    uniq = [str(u) for u in uniq]
    if len(uniq)==2:
        if any(u.upper()=="HGG" for u in uniq):
            pos = [u for u in uniq if u.upper()=="HGG"][0]
            mapping = {uniq[0]: 1 if uniq[0]==pos else 0, uniq[1]: 1 if uniq[1]==pos else 0}
        else:
            mapping = {uniq[0]:0, uniq[1]:1}
        y_num = y_final.map(mapping).astype(int).to_numpy()
    else:
        raise RuntimeError("Cannot encode labels after merge; unexpected categories.")

X_fusion = X_merged.drop(columns=[label_col], errors='ignore')
print("X_fusion shape (with id):", X_fusion.shape)
print("y_final shape:", y_num.shape)
print("Label distribution after merge:", np.unique(y_num, return_counts=True))

X_train = X_fusion.drop(columns=[key_col], errors='ignore')

X_train = X_train.apply(pd.to_numeric, errors='coerce')


nan_rows = X_train.isna().any(axis=1)
if nan_rows.any():
    print(f"Warning: {nan_rows.sum()} rows have NaN after numeric conversion; dropping them.")
    X_train = X_train.loc[~nan_rows].reset_index(drop=True)
    y_num = y_num[~nan_rows.values]
    print("Shapes after dropping NaN rows:", X_train.shape, y_num.shape)


if X_train.shape[0] != y_num.shape[0]:
    raise RuntimeError(f"Mismatch after cleaning: X rows {X_train.shape[0]} vs y length {y_num.shape[0]}")


model =models['Logistic Regression']

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_proba = cross_val_predict(model, X_train, y_num, cv=cv, method='predict_proba')[:, 1]


fpr, tpr, _ = roc_curve(y_num, y_proba)
roc_auc = roc_auc_score(y_num, y_proba)

mean_auc, lower_ci, upper_ci = bootstrap_auc_ci(y_num, y_proba, n_boot=1000, ci=95)

plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Fusion (LR) AUC = {roc_auc:.3f}')
plt.plot([0,1], [0,1], color='navy', linestyle='--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Fusion (Deep + Handcrafted)')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

print(f"\nROC AUC (Fusion LR): {roc_auc:.3f}")
print("Final training feature shape:", X_train.shape)


In [ ]:
# Robust Fusion ROC (Deep + Handcrafted)+PCA using LR

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA


deep_path = "/kaggle/input/deep-features/deep_features.csv"
hand_path = "/kaggle/input/handcrafted-features/Handcrafted_Features_flair.csv"


X_deep = pd.read_csv(deep_path)
X_hand = pd.read_csv(hand_path)

print("Loaded shapes -> deep:", X_deep.shape, ", handcrafted:", X_hand.shape)


key_col = "id"
label_col = "label"


print("\nHandcrafted columns (tail):", X_hand.columns[-10:].tolist())
print("Deep columns (tail):", X_deep.columns[-10:].tolist())


if label_col not in X_hand.columns:
    raise RuntimeError(f"Label column '{label_col}' not found in handcrafted file. Columns: {X_hand.columns.tolist()}")


label_vals = X_hand[label_col].unique()
print("\nLabel unique values:", label_vals)
print("Label counts:\n", X_hand[label_col].value_counts(dropna=False))


y_series = X_hand[label_col].copy()

try:
    y_numeric = pd.to_numeric(y_series, errors='coerce')
    if set(y_numeric.dropna().unique()).issubset({0,1}):
        y_encoded = y_numeric.fillna(y_numeric)  # keep numeric 0/1
        print("Labels are numeric 0/1.")
    else:
        # fallback to mapping strings
        raise ValueError
except Exception:
    # map common strings
    unique_vals = list(pd.Categorical(y_series).categories)
    unique_vals = [str(v) for v in unique_vals]
    print("Detected label categories (strings):", unique_vals)
    if len(unique_vals) == 2:
        # prefer mapping 'HGG'->1 if present
        if any(s.upper() == "HGG" for s in unique_vals):
            pos = [s for s in unique_vals if s.upper() == "HGG"][0]
            mapping = {unique_vals[0]: 1 if unique_vals[0]==pos else 0,
                       unique_vals[1]: 1 if unique_vals[1]==pos else 0}
        elif any(s.upper() == "YES" for s in unique_vals) or any(s.upper()=="POS" for s in unique_vals):
            # heuristic
            mapping = {unique_vals[0]:1, unique_vals[1]:0}
        else:
            # map second category to 1 (arbitrary but consistent)
            mapping = {unique_vals[0]:0, unique_vals[1]:1}
        print("Mapping used for labels:", mapping)
        y_encoded = y_series.map(mapping).astype(int)
    else:
        raise RuntimeError(f"Unexpected number of label categories: {len(unique_vals)} - cannot auto-encode.")

# Final check
if y_encoded.isnull().any():
    raise RuntimeError("After encoding, some labels are NaN. Inspect 'label' column values.")


if key_col in X_deep.columns:
    n_ids = X_deep[key_col].nunique()
    n_rows = X_deep.shape[0]
    print(f"\nDeep has {n_rows} rows and {n_ids} unique ids.")
    if n_rows != n_ids:
        print("Aggregating deep features by id (mean).")
        X_deep_agg = X_deep.groupby(key_col).mean().reset_index()
    else:
        X_deep_agg = X_deep.copy()
else:
    raise RuntimeError(f"Key column '{key_col}' not found in deep features. Columns: {X_deep.columns.tolist()}")

print("Deep (agg) shape:", X_deep_agg.shape)


if key_col not in X_hand.columns:
    raise RuntimeError(f"Key column '{key_col}' not found in handcrafted data. Columns: {X_hand.columns.tolist()}")


hand_for_merge = X_hand[[key_col, label_col]]


X_merged = pd.merge(hand_for_merge, X_deep_agg, on=key_col, how='inner')
print("\nAfter merge (hand w/ deep agg):", X_merged.shape)


y_final = X_merged[label_col].copy()

if pd.api.types.is_numeric_dtype(y_final):
    if set(y_final.unique()).issubset({0,1}):
        y_num = y_final.astype(int).to_numpy()
    else:
        # try to map via earlier mapping if exists
        y_num = pd.to_numeric(y_final, errors='coerce').astype(int).to_numpy()
else:
    # use the mapping we used before (recreate mapping)
    # Recreate mapping logic: map unique two categories -> 0/1
    uniq = list(pd.Categorical(y_final).categories)
    uniq = [str(u) for u in uniq]
    if len(uniq)==2:
        if any(u.upper()=="HGG" for u in uniq):
            pos = [u for u in uniq if u.upper()=="HGG"][0]
            mapping = {uniq[0]: 1 if uniq[0]==pos else 0, uniq[1]: 1 if uniq[1]==pos else 0}
        else:
            mapping = {uniq[0]:0, uniq[1]:1}
        y_num = y_final.map(mapping).astype(int).to_numpy()
    else:
        raise RuntimeError("Cannot encode labels after merge; unexpected categories.")


X_fusion = X_merged.drop(columns=[label_col], errors='ignore')
print("X_fusion shape (with id):", X_fusion.shape)
print("y_final shape:", y_num.shape)
print("Label distribution after merge:", np.unique(y_num, return_counts=True))


X_train = X_fusion.drop(columns=[key_col], errors='ignore')

X_train = X_train.apply(pd.to_numeric, errors='coerce')

nan_rows = X_train.isna().any(axis=1)
if nan_rows.any():
    print(f"Warning: {nan_rows.sum()} rows have NaN after numeric conversion; dropping them.")
    X_train = X_train.loc[~nan_rows].reset_index(drop=True)
    y_num = y_num[~nan_rows.values]
    print("Shapes after dropping NaN rows:", X_train.shape, y_num.shape)


if X_train.shape[0] != y_num.shape[0]:
    raise RuntimeError(f"Mismatch after cleaning: X rows {X_train.shape[0]} vs y length {y_num.shape[0]}")


pipe=Pipeline([("pca", PCA(n_components=189,random_state=42)),("clf",models['Logistic Regression'])])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_proba = cross_val_predict(pipe, X_train, y_num, cv=cv, method='predict_proba')[:, 1]

fpr, tpr, _ = roc_curve(y_num, y_proba)
roc_auc = roc_auc_score(y_num, y_proba)

mean_auc, lower_ci, upper_ci = bootstrap_auc_ci(y_num, y_proba, n_boot=1000, ci=95)

plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Fusion+PCA (LR) AUC = {roc_auc:.3f}')
plt.plot([0,1], [0,1], color='navy', linestyle='--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Fusion+PCA')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

print(f"\n ROC AUC (Fusion+PCA LR): {roc_auc:.3f}")
print("Final training feature shape:", X_train.shape)
